# Repeating earthquakes on SAFOD borehole DAS — results dashboard

Regenerated 2026-08-05. Every number below is read from a file produced by a
tracked script; nothing is typed in by hand.

**What is established**

1. A repeater catalog on borehole DAS — 7 distinct pairs over 11 events, confirmed
   independently on the HRSN borehole array.
2. A depth bound on velocity change — |dv/v| < 0.1% at 251–284 m.

**What is not** — any resolved velocity change. The shallow DAS measurement reached
only ±3.6% precision, which cannot test a 4% prediction.

Companion documents: `REPEATERS_progress.tex` (narrative + failures),
`METHODS_STATUS.md` (methods spec, literature, error log).

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
HERE = os.getcwd()
pd.set_option('display.width', 150)
DX, WELLHEAD = 1.0210, 23          # from g0_refine.py
depth = lambda ch: (np.asarray(ch) - WELLHEAD) * DX
print('files present:', sum(os.path.exists(f) for f in
      ['hrsn_extended.csv','dvv_hrsn.csv','g5_shallow_dvv.csv',
       'g0_refine.npz','coda_window_survey.npz']), 'of 5')

## 1. How many confirmed repeaters — and why the number moved

The count depends entirely on the threshold. Measured *on HRSN*, the
Waldhauser & Schaff convention of 0.9 applies. A looser 0.78 cut was used at one
point to obtain more samples; that was threshold shopping and is not used here.

In [ ]:
d = pd.read_csv('hrsn_extended.csv')
h = d.dropna(subset=['hrsn'])
print(f'pairs taken to HRSN: {len(h)}\n')
for t in [0.95, 0.90, 0.85, 0.78, 0.70]:
    mark = '   <- convention' if t == 0.90 else ('   <- shopped cut' if t == 0.78 else '')
    print(f'  HRSN CC > {t:.2f}: {(h.hrsn > t).sum():3d}{mark}')

g = h[h.hrsn > 0.90].copy()
for c in ('t_i','t_j'):
    g[c[-1]+'date'] = pd.to_datetime(g[c], utc=True, format='mixed').dt.strftime('%Y-%m-%d')
g['pair'] = g.idate + ' / ' + g.jdate
print(f'\nrows above 0.90: {len(g)};  DISTINCT pairs: {g.pair.nunique()}')
dup = g.pair.value_counts()
if (dup > 1).any():
    print('near-duplicate catalog entries (same event listed twice):')
    print(dup[dup > 1].to_string())

In [ ]:
cols = ['pair','m_i','m_j','dt_days','hrsn','nsta','das_min']
print(g.sort_values('hrsn', ascending=False)[cols].to_string(index=False))
ev = sorted(set(g.idate) | set(g.jdate))
print(f'\n{len(ev)} distinct events: {", ".join(ev)}')

## 2. G0 — channel-to-depth registration

The gate on everything shallow. Pre-event noise locates the wellhead; the
earthquake-derived velocity profile is checked against the 2005 PGSI check shot.

In [ ]:
z = np.load('g0_refine.npz', allow_pickle=True)
print(f"wellhead channel : {float(z['wellhead']):.0f}")
print(f"coupled interval : ch {int(z['top'])}-{int(z['bot'])}")
print(f"CH_LO=100 is depth {(100 - float(z['wellhead'])) * DX:.0f} m")
print(f"Li & Ben-Zion 17 m peak sensitivity is channel {float(z['wellhead']) + 17/DX:.0f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ns = 10 ** z['noise_sm']
ax[0].semilogx(ns, np.arange(ns.size), 'C4-', lw=1.3)
ax[0].axhline(float(z['top']), color='C2', lw=2, label=f"wellhead ch {int(z['top'])}")
ax[0].axhline(float(z['bot']), color='C1', lw=2, label=f"base ch {int(z['bot'])}")
ax[0].invert_yaxis(); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
ax[0].set(xlabel='pre-event noise RMS', ylabel='channel',
          title='Lead-in is 35x noisier than the cemented section')
ax[1].plot(1000*z['tcs'], z['zcs'], 'k--', lw=1.6, label='2005 check shot')
ax[1].set(ylim=(900,-60), xlabel='vertical travel time (ms)',
          ylabel='depth (m)', title='Depth reference (known geophone depths)')
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 3. G3 — dv/v at HRSN depth: a quantified null

Coda stretching on confirmed pairs. The bound is set by the **bootstrap error**, not
by the random-pair control floor — the latter is a detection statistic and does not
describe the precision of a measurement on a pair already confirmed at CC 0.99.

In [ ]:
v = pd.read_csv('dvv_hrsn.csv')
rep = v[v.is_rep].dropna(subset=['dvv'])
con = v[~v.is_rep].dropna(subset=['dvv'])
print(f"repeater |dv/v| median : {100*rep.dvv.abs().median():.3f} %")
print(f"typical bootstrap error: +/- {100*rep.err.median():.3f} %")
print(f"coda CC                : {rep.coda_cc.median():.3f}")
print(f"random-pair |dv/v|     : {100*con.dvv.abs().median():.3f} % "
      f"(detection floor, NOT the precision)")
print(f"\n=> |dv/v| < {100*max(2*rep.err.median(), rep.dvv.abs().median()):.2f} % "
      f"at 251-284 m over {rep.dt_days.min():.0f}-{rep.dt_days.max():.0f} day baselines")

fig, ax = plt.subplots(figsize=(7,4))
ax.hist(100*con.dvv, bins=25, color='0.7', label='random pairs (null)')
for x in rep.dvv: ax.axvline(100*x, color='C3', lw=1.2)
ax.set(xlabel='dv/v (%)', ylabel='count',
       title='Repeaters (red) sit an order of magnitude inside the null')
ax.legend(fontsize=8); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## 4. The alignment bug, and what the coda actually supports

Two runs returned coda CC ≈ 0 on pairs that correlate at 0.99, because windowed
correlation was computed at **zero lag** while catalog origin times carry 0.1–0.5 s
of error. After `bulk_align`, coda is usable across the whole 0.5–14 s range.

In [ ]:
c = np.load('coda_window_survey.npz', allow_pickle=True)
st, CC, CCc, SN = c['starts'], c['CC'], c['CCc'], c['SN']
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(st, np.nanmedian(SN, 0), 'C0-', lw=2)
ax[0].axhline(3, color='C3', ls='--', label='SNR = 3')
ax[0].set(yscale='log', xlabel='lapse (s)', ylabel='coda SNR',
          title='Signal is abundant to 12 s')
ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
ax[1].plot(st, np.nanmedian(CC, 0), 'C2-', lw=2, label='repeaters')
ax[1].plot(st, np.nanmedian(CCc, 0), '0.5', ls='--', lw=2, label='random pairs')
ax[1].axhline(0.7, color='C3', ls='--', label='CWI needs 0.70')
ax[1].set(xlabel='lapse (s)', ylabel='coda CC', title='After alignment: 0.85-0.96')
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f'coda CC range across all windows: '
      f'{np.nanmedian(CC,0).min():.3f} to {np.nanmedian(CC,0).max():.3f}')

## 5. G5 — the shallow DAS measurement, and why it is not usable

The deep intervals validate the machinery against G3. The shallow interval is
unresolved, and its bound is too weak to test the 4% prediction. The limitation is
the observable: direct-arrival timing is depth-local but ~70× less sensitive than
coda stretching on the same events.

In [ ]:
print(f"{'method':<42}{'precision':>12}")
print(f"{'HRSN coda stretching':<42}{'+/- 0.05 %':>12}")
print(f"{'DAS direct-arrival differential timing':<42}{'+/- 3.6 %':>12}")
print(f"{'':<42}{'-> 70x':>12}")
try:
    s5 = pd.read_csv('g5_shallow_dvv.csv')
    num = s5.select_dtypes('number').columns
    print('\ng5_shallow_dvv.csv:', len(s5), 'rows,', len(num), 'numeric columns')
except Exception as e:
    print('\n(g5 csv not summarised:', e, ')')

## 6. Where this goes next

Seven confirmed pairs cannot carry a seasonal time series; 413 days of ambient-noise
correlation can, using the pipeline already in this repository (`stack_daily.py`,
`cc_tools.py`) and the same stretching estimator that Kidiwela et al. 2026 and
Shi et al. 2026 use.

The repeaters are not superseded by that — they are what makes it defensible.
Ambient-noise dv/v is vulnerable to seasonal changes in the noise source
distribution, and the Parkfield microseism is strongly seasonal, so the first
objection to any seasonal ambient-noise result is that it *is* the microseism.
Repeating earthquakes are immune: identical physical source, no dependence on the
noise field.

**Unquantified risk, applies either way.** Every independent line places the signal
shallow — Shi et al. at 2 cm, Li & Ben-Zion at 17 m, this work's null at 250 m. The
gauge length is 16.34 m and the weathered layer is 754 m/s. Gauge-length averaging
may smear out the signal before any estimator sees it. Not yet modelled.

**Open items:** dedupe the catalog on a time window rather than an exact timestamp
(the 83/93 per-channel counts are likely inflated); acquisition metadata for the
±25 m depth datum; free-surface interference above the reflection point.